In [ ]:
import os
import cv2
import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"OpenCV Version: {cv2.__version__}")

In [ ]:
# Target data directory (checks both notebook-relative and project-root-relative)
data_dir = "../../data/raw/mvtec_itodd"
if not os.path.exists(data_dir):
    data_dir = "data/raw/mvtec_itodd"

rgb_img = None
sample_path = None

# Search recursively for any TIF or PNG files (skipping hidden cache directories)
print(f"Searching recursively for a sample TIF/PNG image in: {os.path.abspath(data_dir)}...")
img_files = []
for root, dirs, files in os.walk(data_dir):
    # Skip hidden directories (like the HuggingFace .cache folder)
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    for file in files:
        if file.lower().endswith((".tif", ".tiff", ".png")):
            img_files.append(os.path.join(root, file))

if img_files:
    # Sort files to get a consistent sample
    img_files.sort()
    # Pick the first 2D camera image (usually ending in '_l.tif') if available, otherwise the first image
    sample_path = next((f for f in img_files if f.endswith("_l.tif")), img_files[0])
    print(f"Found sample image: {sample_path}")
    
    # Load and process image
    try:
        bgr_img = cv2.imread(sample_path, cv2.IMREAD_UNCHANGED)
        if bgr_img is not None:
            # If the image is grayscale (1 channel), duplicate channels for RGB mapping
            if len(bgr_img.shape) == 2:
                rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_GRAY2RGB)
            else:
                rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
        else:
            print(f"OpenCV failed to read the file: {sample_path}")
    except Exception as e:
        print(f"Failed to read image: {e}")
else:
    print("\n[!] No TIF, TIFF, or PNG files found in the dataset directory.")
    print("Please make sure you have extracted the dataset archives first by running:")
    print("  just extract-data")

# Process and visualize the image if loaded
if rgb_img is not None:
    # Transform to deep learning tensor layout (C, H, W) normalized between [0.0, 1.0]
    tensor_transform = transforms.ToTensor()
    img_tensor = tensor_transform(rgb_img)
    print(f"Successfully processed image into Tensor Shape: {img_tensor.shape}")
    
    # Plot the sample
    plt.figure(figsize=(8, 6))
    plt.imshow(rgb_img)
    plt.title(f"MVTec ITODD Sample Exploration\n({os.path.basename(sample_path)})")
    plt.axis("off")
    plt.show()